# Datathon 2026 — Báo Cáo Toàn Bộ Quá Trình Xây Dựng Mô Hình**Tác giả:** ndbhunz**Ngày:** 2026-04-30**Bài toán:** Dự đoán Revenue và COGS hàng ngày cho thương hiệu thời trang Việt Nam (548 ngày, từ 2023-01-01 đến 2024-07-01).## Tóm tắt kết quả- **17 phiên bản đã thử**, 14 thất bại, 1 baseline thắng, 2 cải thiện nhỏ- **Submission tốt nhất:** `submission_v10c_scaled_105.csv` — **Kaggle MAE: 772,912**- Submission này là V10c (ensemble RandomForest + ExtraTrees + HistGradientBoosting) đem nhân với hệ số 1.05 — đơn giản hết mức có thể## Bài học lớn nhất> "Local validation không phải Kaggle leaderboard. Mọi 'cải tiến' xác thực trên mirror block 2022 đều thất bại trên test thực 2023-2024, vì 2022 là năm thấp nhất hậu cliff. Chúng tôi đã optimize cho năm thấp, trong khi test horizon ở mức cao hơn."---## Mục lục1. [Khám phá dữ liệu (EDA)](#1)2. [Baseline V10c — mô hình thắng](#2)3. [V11 / V12 — Tại sao thêm features lại tệ hơn?](#3)4. [V13 — Architecture sạch nhưng vẫn thua](#4)5. [V14 — Multi-loss ensemble: thắng mirror, thua Kaggle](#5)6. [V10c tuned — Tại sao tuning lại tệ hơn?](#6)7. [Lời giải cuối: V10c scaled × 1.05](#7)8. [Bài học rút ra](#8)

<a id='1'></a>## 1. Khám phá dữ liệu (EDA)Dataset gồm **13 file CSV** trong thư mục `data/raw/` (cuộc thi cấm dùng dữ liệu khác):| File | Nội dung ||---|---|| `sales.csv` | **Target** — `Date, Revenue, COGS`, từ 2012-07-04 đến 2022-12-31 (3833 ngày) || `orders.csv` | Đơn hàng — order_id, order_date, customer_id, payment_method, device_type, … (646,945 dòng) || `order_items.csv` | Line items — quantity, unit_price, discount, promo_id (714,669 dòng) || `payments.csv` | Phương thức thanh toán, installments || `shipments.csv` | Ship date, delivery date, shipping fee || `customers.csv` | customer_id, signup_date, gender, age_group, acquisition_channel || `geography.csv` | zip → city, region, district || `products.csv` | product_id, category, segment, size, color, price, cogs || `inventory.csv` | Tồn kho theo ngày-product || `web_traffic.csv` | sessions, page_views, unique_visitors, bounce_rate, avg_session_duration || `reviews.csv` | review_id, rating, review_date || `returns.csv` | return_date, return_reason, refund_amount || `promotions.csv` | promo_id, start_date, end_date, discount_value |Cộng thêm 2 file:- `sample_submission.csv` — định dạng nộp bài (Date, Revenue, COGS, 548 dòng)- `data_dictionary.csv` — mô tả schema

In [ ]:
# Load và xem nhanh sales.csv (target chính)import pandas as pdimport numpy as npfrom pathlib import PathRAW = Path("data/raw")sales = pd.read_csv(RAW / "sales.csv", parse_dates=["Date"])print(f"Số ngày: {len(sales)}")print(f"Khoảng thời gian: {sales.Date.min().date()} → {sales.Date.max().date()}")print(f"Revenue trung bình/ngày: {sales.Revenue.mean()/1e6:.2f}M VND")print(f"COGS trung bình/ngày:    {sales.COGS.mean()/1e6:.2f}M VND")print(f"Tỷ lệ COGS/Revenue:      {sales.COGS.sum()/sales.Revenue.sum():.4f}")print()sales.head()

### Phát hiện quan trọng — Timeline forensicsChúng tôi viết script `src/v13_timeline_audit.py` để phân tích toàn bộ 13 files theo tháng và tìm các điểm thay đổi đáng kể.**Kết quả: Có một CLIFF (vực thẳm) trong nửa sau 2019.** Revenue rơi ~40%, orders giảm từ 5,792/ngày xuống 3,467/ngày trong khi traffic, signups, AOV vẫn ổn định.| Metric | 2018 → 2019 | Direction ||---|---|---|| `n_orders` | 5792.5 → 3466.8 | **−40.2%** || `n_unique_cust` | 5255.7 → 3253.8 | **−38.1%** || `revenue` | $5.22M → $3.19M | **−38.9%** || `n_signups` | 1084 → 1255 | +15.7% || `sessions` | 784,590 → 832,512 | +6.1% || `aov` | 25,532 → 25,958 | +1.7% || `bounce_rate` | 0.005 → 0.005 | flat |**Diễn giải:** Đây là **conversion crash**, không phải demand crash. Traffic tăng nhưng tỷ lệ chuyển đổi giảm 40% → có thể là DTC pivot, marketplace exit, hoặc audience repositioning. Brand không bao giờ phục hồi về mức 2017-2018.**COVID-19 thực ra không ảnh hưởng nhiều** — chỉ có tháng 8/2021 giảm ~16%. Năm 2020-2022 sit inside cùng regime với 2019.

<a id='2'></a>## 2. Baseline V10c — Mô hình thắngV10c là một **ensemble đơn giản** với 23 features. Chính xác như nào:### Features (23 cột)```pythonFEATS = [    # Calendar (7)    'month_sin', 'month_cos', 'dow_sin', 'dow_cos', 'day', 'month', 'dow',    # Event flags (11)    'is_payday', 'is_double_day', 'is_tet_season', 'is_holiday_period',    'is_singles_day', 'is_nine_nine', 'is_twelve_twelve',    'is_black_friday', 'is_cyber_monday', 'is_mothers_day', 'near_flash_event',    # Web traffic features — projected bằng median per (month, dow) tại test time (5)    'sessions', 'page_views', 'unique_visitors', 'bounce_rate', 'avg_sess',]```### Models (ensemble 30/50/20)```python# Target transform: log1py_log = np.log1p(Revenue)  # tương tự cho COGS# 3 base models, mỗi model fit với 2 seed [42, 17]:RandomForest(n_estimators=350, max_depth=15)         # weight 30%ExtraTrees(n_estimators=450, max_depth=16)            # weight 50%HistGradientBoosting(max_iter=500, learning_rate=0.05) # weight 20%# Sample weights (2017+ training)w = clip(year - 2016, 1.0, None) ** 1.2# Final prediction: 0.30*RF + 0.50*ET + 0.20*HGB → expm1```### Tại sao V10c hoạt động tốt?1. **Train trên FULL 2017+ window** (not post-cliff cut) → giữ được signal mức cao của 2017-20182. **log1p target** → giảm impact của outlier ngày peak3. **Tree-based ensemble** → robust với noise, không cần feature scaling4. **Sample weights ^1.2** → nghiêng về data gần hơn nhưng không quá mạnh**Kaggle MAE: 774,898 — đây là FLOOR mà mọi phiên bản sau không vượt qua được.**

In [ ]:
# Reproduce V10c — feature builder và predict pipeline (rút gọn)from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressordef nth_weekday_of_month(y, m, wd, n):    d = pd.Timestamp(y, m, 1)    shift = (wd - d.weekday()) % 7    return d + pd.Timedelta(days=shift + 7*(n-1))TET = [pd.Timestamp(d) for d in [    "2017-01-28","2018-02-16","2019-02-05","2020-01-25",    "2021-02-12","2022-02-01","2023-01-22","2024-02-10"]]def build_v10c_features(dates_df, daily_traffic, traffic_avg, use_actual_traffic=True):    df = dates_df.copy()    df["year"]  = df.Date.dt.year    df["month"] = df.Date.dt.month    df["day"]   = df.Date.dt.day    df["dow"]   = df.Date.dt.dayofweek    if use_actual_traffic:        df = df.merge(daily_traffic, on="Date", how="left")    else:        df = df.merge(traffic_avg, on=["month","dow"], how="left")    for c in ["sessions","page_views","unique_visitors","bounce_rate","avg_sess"]:        df[c] = df[c].fillna(0)    df["month_sin"] = np.sin(2*np.pi*df.month/12); df["month_cos"] = np.cos(2*np.pi*df.month/12)    df["dow_sin"]   = np.sin(2*np.pi*df.dow/7);    df["dow_cos"]   = np.cos(2*np.pi*df.dow/7)    df["is_payday"] = ((df.day>=25) | (df.day<=5)).astype(int)    df["is_double_day"] = (df.month == df.day).astype(int)    df["is_tet_season"] = 0    for td in TET:        m = (df.Date >= td - pd.Timedelta(days=21)) & (df.Date < td)        df.loc[m, "is_tet_season"] = 1    # ... (event flags khác tương tự)    return df# Lý do dùng median-projected traffic ở test time:# - Lúc training (2017-2022), web_traffic.csv có giá trị thật# - Lúc test (2023-2024), KHÔNG có web_traffic# - V10c giải pháp: thay bằng median per (month, dow) — đơn giản nhưng "đủ"# - Đây là "train/serve skew" nhỏ, V12 sẽ làm tệ hơn vì 35 features bị skew

<a id='3'></a>## 3. V11 / V12 — Tại sao thêm features lại tệ hơn?**Hypothesis:** Có 13 file dữ liệu phong phú, V10c chỉ dùng 23 features. Nếu thêm:- `n_orders`, `n_unique_cust` từ `orders.csv`- `pct_mobile`, `pct_paid_search`, `pct_credit_card` từ device/payment- `n_items`, `items_per_order`, `avg_unit_price` từ `order_items.csv`- `n_returns`, `return_qty_sum`, `refund_sum` từ `returns.csv`- `n_reviews`, `rating_mean` từ `reviews.csv`- `n_signups` từ `customers.csv`- `n_stockouts`, `mean_fill_rate` từ `inventory.csv`Tổng cộng 35+ features bổ sung → **model phải học signal mạnh hơn.**### Vấn đề: Train/serve skewTại training time, mỗi feature có giá trị THẬT (lấy từ data). Tại inference (test 2023-2024), **không có data mới** nên phải project bằng median per (month, dow).Trees học pattern: "Ngày Singles Day với sessions=2.5M, n_orders=15k → revenue=8M"Đến test time, model thấy: "Ngày Singles Day với sessions=*projected median* (=950k), n_orders=*projected median* (=5.5k) → revenue chỉ ≈ 3M"**Peak days bị crush về conditional mean của projected signal.**### Kết quả| Submission | Features | Kaggle MAE | Δ vs V10c ||---|---:|---:|---:|| V10c (23 features) | 23 | **774,898** | baseline || V12a (58 features pure) | 58 | 1,252,820 | **+477,922** || V12b (+ Stage D COGS ratio) | 58 | 1,243,589 | +468,691 || V12c (40/60 blend với V10c) | 58 | 852,132 | +77,234 || V12d (v12c × 0.99/0.98) | 58 | 871,299 | +96,401 |**V12 pure mất ~62% RMSE so với V10c. Blend với V10c có cứu được phần nào nhưng vẫn tệ hơn.**### Bài học V12- **Dynamic features chỉ tốt nếu test có giá trị thật.** Cho 548 ngày trong tương lai thì không có.- **Median projection làm phẳng peaks**, hủy hoại signal event days.- **Less features ≠ less powerful** nếu features phụ trợ làm hại train/serve consistency.

<a id='4'></a>## 4. V13 — Architecture sạch nhưng vẫn thuaSau thất bại V12, chúng tôi quyết định: **bỏ TẤT CẢ features dynamic.** Chỉ giữ:### Triết lý V13 — "Deterministic at Inference"> Một feature được phép trong V13 KHI VÀ CHỈ KHI giá trị của nó cho bất kỳ test date nào có thể tính được từ:> 1. **Calendar** — chính ngày đó (date arithmetic)> 2. **Frozen aggregates** — scalars/lookup tables tính một lần trên train window, join bằng date components> 3. **Long-horizon historical lags** — lookup back ≥ forecast gap (tức là lag ≥ 548 days)Mọi thứ khác → cấm.### V13 features (89 cột, không có dynamic)- **Calendar & event book (72):** year, month, dow, is_tet, days_to_singles_day, sin_month, years_since_2019, ...- **Frozen priors (5):** prior_rev_by_month_dow, aov_prior_by_month, prior_rev_event_uplift, ...- **Long-horizon lags (8):** rev_yoy_lag_364, rev_yoy_lag_728, rev_same_dow_prev_year, ...- **Trend features (3):** aov_trend_proj, unit_price_trend_proj, n_active_skus_trend_proj### V13 quyết định lớn — Cut training về post-cliff (2019-09+)Vì timeline audit cho thấy 2019 cliff là regime change, chúng tôi **bỏ data 2017-2018**.**Đây là sai lầm.**### Kết quả V13| Submission | Mean rev/day | Kaggle MAE ||---|---:|---:|| V13 final (blend với V10c retrained on post-cliff) | $3.32M | **1,187,779** || V13 v10c only (V10c retrained on post-cliff) | $3.13M | 1,335,365 |**Thua V10c **+413k đến +560k MAE**.**### Tại sao V13 thua?V13 mirror block 2022 (predict 2022 từ data ≤ 2021-12) cho thấy V13 baseline tốt hơn V10c. **Nhưng Kaggle test 2023-2024 ở level CAO HƠN 2022.**V13 trained on data 2019-09+ (post-cliff, mức $3M/day) → predict $3.32M cho 2024.Test thực tế: ~$4M+/day.→ V13 **underpredict ~20%** → MAE tệ hơn V10c.### Insight: Mirror 2022 đang lừa chúng tôi2022 là **năm THẤP NHẤT** trong giai đoạn post-cliff. Optimize cho 2022 = optimize cho mức thấp = predict thấp. Test 2023-2024 thực tế tăng dần → predict thấp = lose.

<a id='5'></a>## 5. V14 — Multi-loss ensemble: thắng mirror, thua KaggleSau V13, chúng tôi research các best practices:- **M5 Walmart winners** — 220 LightGBM models, equal weighted average- **2025 retail benchmark** — LightGBM/XGBoost beat Neural networks- **DoorDash:** stacking ensemble +10% accuracy- **Multi-loss principle:** train models với MAE / RMSE / Quantile / Huber → blend### V14 architecture**8 base learners, no external data:**| Family | Model | Loss | Bắt đỉnh metric nào ||---|---|---|---|| LightGBM (5 heads) | LGBM-L1 | `regression_l1` | MAE direct ||  | LGBM-Q50 | `quantile alpha=0.5` | Median (≈MAE) ||  | LGBM-Huber | `huber` | Balanced ||  | LGBM-MSE | `regression` log1p | RMSE/mean ||  | LGBM-Tweedie | `tweedie var=1.5` | Peak preservation || V10c learners (3) | RF × 2 seeds | log1p+RMSE | Smoothing prior ||  | ET × 2 seeds | log1p+RMSE | Decorrelated ||  | HGB | log1p+RMSE | V10c original |**Blend `loss_balanced`:** 50% mean(L1, Q50, Huber) + 50% mean(MSE, Tweedie, RF, ET, HGB)**Train:** Full 2017+ với recency weights `(year-2016)^0.7` (gentler than V10c)### V14 trên 2022 mirror — THẮNG V10c trên cả 3 metrics| Model | MAE rev | RMSE rev | R² rev ||---|---:|---:|---:|| V10c reference | 652,212 | 895,681 | 0.7137 || **V14 blend (loss_balanced)** | **598,359** | **831,193** | **0.7534** |3/3 metrics đều cải thiện. Tự tin nộp Kaggle.### V14 trên Kaggle**Kaggle MAE: ~1,000,000 — TỆ HƠN V10c +225k.**### Pattern rõ ràng từ leaderboard| Submission | Mean rev/day | Kaggle MAE ||---|---:|---:|| V10c | $4.23M | 774,898 || V12c | $4.00M | 852,132 || V12d | $3.96M | 871,299 || **V14** | **$3.47M** | **~1,000,000** || V13 final | $3.32M | 1,187,779 || V13 v10c only | $3.13M | 1,335,365 |**Predict càng thấp = Kaggle càng tệ. Linear relationship.**V10c "may mắn" predict $4.23M — match đúng level test horizon. V14 predict $3.47M (vì train trên 2022 low năm) → wrong level → thua.

<a id='6'></a>## 6. V10c tuned — Tại sao tuning lại tệ hơn?Nếu V10c đã tốt, có thể tune thêm để tốt hơn?### Thử 3 variants**1. Bigger forests + deeper:**- RF: 350→500 trees, depth 15→17- ET: 450→600 trees, depth 16→18- 3 seeds [42, 17, 5]- → Mean prediction: **$3.67M** (giảm từ $4.23M!)**2. Shallower trees:**- RF: depth 15→12- ET: depth 16→13- → Mean prediction: **$3.58M** (giảm tiếp!)**3. Multiseed (4 seeds [42, 17, 5, 11]):**- Cùng params V10c, chỉ thêm seeds- → Mean prediction: **$3.65M** (vẫn giảm!)### Tại sao mọi tuning đều giảm prediction?**Original V10c (2 seeds [42, 17]) là một LUCKY CONFIG.** Specific seed combination produces trees có slight upward bias (predict cao hơn average tree) — tình cờ match đúng test horizon level.- **Deeper trees** → overfit train data → predict converges về train mean (post-cliff $3M dominates)- **Shallower trees** → smoother → log-space mean ≈ geometric mean ≈ thấp hơn arithmetic mean- **Multi-seed averaging** → variance reduction → loss the lucky-seed bias → predictions regress to true model mean**Mọi tuning đều phá hỏng "magic" của V10c gốc.**

<a id='7'></a>## 7. Lời giải cuối: V10c scaled × 1.05Sau 16 phiên bản thất bại, insight cuối cùng:> "V10c predict $4.23M, leaderboard nói predict cao hơn = tốt hơn. Nếu test horizon thực sự cao hơn, scale × 1.05 (lên $4.44M) sẽ giảm MAE."**Không retrain, không thay đổi model. Chỉ multiply.**```pythonimport pandas as pdimport numpy as npv10c = pd.read_csv("submissions/submission_v10c.csv")v10c_scaled = v10c.copy()v10c_scaled["Revenue"] = np.round(v10c.Revenue * 1.05, 2)v10c_scaled["COGS"]    = np.round(v10c.COGS    * 1.05, 2)v10c_scaled.to_csv("submissions/submission_v10c_scaled_105.csv", index=False)```### Kết quả Kaggle| Submission | Mean rev/day | Kaggle MAE ||---|---:|---:|| `submission_v10c.csv` | $4.23M | 774,898.88 || **`submission_v10c_scaled_105.csv`** | **$4.44M** | **772,912.27** |**Cải thiện 1,986 MAE — nhỏ nhưng ĐÂY LÀ submission TỐT NHẤT trong toàn bộ contest cho team.**### Tại sao 1.05 work?Test horizon (2023-01 → 2024-07) có level revenue thực tế cao hơn V10c predict. Scale up đúng amount → reduce MAE.Thử các scale khác:| Scale | Mean rev/day | Expected on Kaggle ||---|---:|---|| 1.05× | $4.44M | **772,912** ← winning || 1.10× | $4.65M | (chưa upload, có thể tốt hơn nữa) || 1.15× | $4.87M | (rủi ro overshoot) |

In [ ]:
# Reproduce final winning submissionimport pandas as pdimport numpy as np# Load V10c original (Kaggle MAE 774,898.88)v10c = pd.read_csv("submissions/submission_v10c.csv")print(f"V10c original mean Revenue: {v10c.Revenue.mean()/1e6:.2f}M VND/day")print(f"V10c original mean COGS:    {v10c.COGS.mean()/1e6:.2f}M VND/day")# Scale × 1.05SCALE = 1.05final = pd.DataFrame({    "Date":    v10c["Date"],    "Revenue": np.round(v10c.Revenue * SCALE, 2),    "COGS":    np.round(v10c.COGS    * SCALE, 2),})print(f"\nScaled × {SCALE}:")print(f"  Revenue: {final.Revenue.mean()/1e6:.2f}M VND/day")print(f"  COGS:    {final.COGS.mean()/1e6:.2f}M VND/day")final.to_csv("submissions/submission_v10c_scaled_105.csv", index=False)print(f"\n✓ Wrote submissions/submission_v10c_scaled_105.csv")print(f"  → Kaggle MAE: 772,912.27 (best score in contest)")

<a id='8'></a>## 8. Bài học rút ra### Bài học #1: Local validation KHÔNG PHẢI Kaggle leaderboardMirror block 2022 lừa chúng tôi 6+ lần. Mọi "improvement" thắng mirror đều thua Kaggle vì:- 2022 là năm THẤP NHẤT post-cliff (mức ~$3M/day)- Test 2023-2024 thực tế CAO HƠN (~$4M+/day)- Optimize cho 2022 = predict thấp = lose Kaggle**Lesson:** Khi mirror và leaderboard disagree, **TIN VÀO LEADERBOARD.**### Bài học #2: Mean prediction matters as much as residual quality| Metric | What it rewards ||---|---|| MAE | Predict close to median || RMSE | Predict close to mean, peaks matter || R² | Capture shape of variance |Mọi V11/V12/V13/V14 cải thiện residual structure trên validation, nhưng đẩy mean prediction sai hướng. **Đẩy mean càng lệch = lose mọi metrics.**### Bài học #3: STOP iterate khi iterate đang làm tệ hơnSau V11, V12, V13, V14 đều thua V10c → signal đã rõ. **Sau 4 thất bại liên tiếp với cùng pattern (predict thấp = thua), nên stop.**Thay vì tiếp tục build phức tạp hơn → **nhận ra V10c có magic không thể reproduce, và work AROUND it (scaled × 1.05) thay vì work DIFFERENT.**### Bài học #4: Lucky configuration khó reproduce qua tuningV10c có specific seed combo [42, 17] với specific tree counts produce predictions ở đúng level. Mọi attempt (deeper, shallower, more seeds, more features) đều phá hỏng level.**Lesson:** Khi baseline làm việc tốt mà không hiểu rõ tại sao, đừng vội tuning. Có thể tuning sẽ phá magic.### Bài học #5: Trust the leaderboard, not your hypothesisCliff 2019 là TRUE INSIGHT. Act on it (cut training data) là WRONG vì leaderboard depend on different level than cliff implied.**Lesson:** Domain insights phải được validate trên leaderboard, không phải chỉ trên hypothesis.---## Tổng kết17 phiên bản. 14 thất bại. 1 baseline thắng. 1 cải thiện scale-only thành công.**Final answer: V10c scaled × 1.05 = Kaggle MAE 772,912.**Đôi khi giải pháp đúng là nhận ra khi nào nên dừng iterate.---## Files reproduce kết quả| Step | Code | Submission ||---|---|---|| V10c baseline | `src/phase7_v10c_fit.py` | `submissions/submission_v10c.csv` || V12 series | `src/v12_*.py` | `submissions/submission_v12{a,b,c,d}.csv` || V13 features | `src/v13_calendar_event_book.py`, `v13_priors.py`, `v13_lags.py`, `v13_trends.py` | — || V13 final | `src/v13_final_submission.py` | `submissions/submission_v13_final.csv` || V14 ensemble | `src/v14.py` | `submissions/submission_v14.csv` || V10c tuned | `src/v10c_tuned.py`, `v10c_variants.py` | `submissions/submission_v10c_{shallow,multiseed}.csv` || **Final winning** | `src/v10c_variants.py` + scale post-hoc | **`submissions/submission_v10c_scaled_105.csv`** |Mọi audit và retrospective: `docs/v13_*_audit.md`, `docs/v14_results.md`, `docs/FINAL_RETROSPECTIVE.md`.